In [2]:
import zipfile
import geopandas as gpd
import os
import locale
import requests
from ee_s1_ard import S1ARDImageCollection
import pandas as pd
import ee

credentials = ee.ServiceAccountCredentials(None, os.environ["GEE"])
ee.Initialize(credentials)

In [3]:
def load_aoi_from_shapefile(shapefile_path):
    gpd_aoi = None

    if shapefile_path.endswith(".zip"):
        with zipfile.ZipFile(shapefile_path, "r") as zip_ref:
            shapefile_within_zip = None
            for file in zip_ref.namelist():
                if file.lower().endswith(".shp"):
                    shapefile_within_zip = file
                    break
            if not shapefile_within_zip:
                raise FileNotFoundError(f"No .shp file found inside the zip archive: {shapefile_path}")
            gpd_aoi = gpd.read_file(f"zip://{shapefile_path}/{shapefile_within_zip}")
    else:
        gpd_aoi = gpd.read_file(shapefile_path)

    gpd_aoi = gpd_aoi.to_crs(epsg=4326)

    if gpd_aoi.empty:
        raise ValueError(f"The shapefile at {shapefile_path} does not contain any geometries.")

    if len(gpd_aoi) > 1:
        gpd_aoi = gpd_aoi.dissolve()

    if gpd_aoi.empty:
        raise ValueError(f"The shapefile at {shapefile_path} became empty after dissolve.")

    geometry = gpd_aoi.geometry.iloc[0]
    geojson = geometry.__geo_interface__

    if geojson["type"] == "Polygon":
        geojson["coordinates"] = [
            list(map(lambda coord: coord[:2], ring)) for ring in geojson["coordinates"]
        ]
    elif geojson["type"] == "MultiPolygon":
        geojson["coordinates"] = [
            [list(map(lambda coord: coord[:2], ring)) for ring in polygon]
            for polygon in geojson["coordinates"]
        ]

    ee_geometry = ee.Geometry(geojson)
    feature = ee.Feature(ee_geometry)
    return ee.FeatureCollection([feature])

In [4]:
aoi = load_aoi_from_shapefile('contorno_area_total.zip')

In [5]:
start_date = '2025-02-09'
stop_date = '2025-08-09'

In [8]:
processor = S1ARDImageCollection(
    geometry=aoi,
    start_date=start_date,
    stop_date=stop_date,
    polarization="VVVH",
    apply_border_noise_correction=True,
    apply_terrain_flattening=True,
    apply_speckle_filtering=True,
    output_format="DB",
    ascending=False # False for descending orbit, True for ascending orbit
)

collection = processor.get_collection().sort("system:time_start", False)  # Sort by time in descending order

collection.size().getInfo()  # This will trigger the processing and return the size of the collection

24

In [9]:
def add_vvvh_ratio_band(image):
    ratio = image.select("VV").divide(image.select("VH")).rename("VVVH_ratio")
    return image.addBands(ratio)

collection = collection.map(add_vvvh_ratio_band)
# Extract the VVVH_ratio band and reduce over the geometry to get mean values for each image
def get_vvvh_ratio_mean(image):
    stats = image.select("VVVH_ratio").reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=10,
        maxPixels=1e9
    )
    # Get image date
    date = image.date().format('YYYY-MM-dd')
    return ee.Feature(None, {
        'date': date,
        'VVVH_ratio_mean': stats.get('VVVH_ratio')
    })

# Map over the collection to get time series
vvvh_ratio_ts = collection.map(get_vvvh_ratio_mean).getInfo()

# Convert to pandas DataFrame for easier handling

data = [
    {'dates': f['properties']['date'], 'AOI_average': f['properties']['VVVH_ratio_mean']}
    for f in vvvh_ratio_ts['features']
]

df = pd.DataFrame(data)
print(df)

         dates  AOI_average
0   2025-08-05     0.681253
1   2025-07-30     0.686562
2   2025-07-24     0.705732
3   2025-07-18     0.693760
4   2025-07-12     0.696250
5   2025-07-06     0.672604
6   2025-06-30     0.645482
7   2025-06-24     0.602126
8   2025-06-18     0.574899
9   2025-06-12     0.600902
10  2025-06-06     0.574432
11  2025-05-31     0.531584
12  2025-05-25     0.565499
13  2025-05-19     0.538727
14  2025-05-13     0.549288
15  2025-05-01     0.499650
16  2025-04-25     0.527878
17  2025-04-19     0.523483
18  2025-04-13     0.551375
19  2025-04-07     0.567859
20  2025-03-26     0.541693
21  2025-03-14     0.574422
22  2025-03-02     0.554751
23  2025-02-18     0.495828


In [10]:
import plotly.express as px

fig = px.line(df, x='dates', y='AOI_average', markers=True, title='VV/VH Ratio Mean Time Series')
fig.update_layout(xaxis_title='Date', yaxis_title='VV/VH Ratio Mean')
fig.show()

In [11]:
from datetime import datetime, timedelta
date = df.dates[3]
next_date = (datetime.strptime(date, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")
selected_image = collection.filterDate(date, next_date).first().select(['VV', 'VH', 'VVVH_ratio']).clip(aoi)

In [12]:
try:
    # Prepare download URL for the clipped image
    url = selected_image.getDownloadURL({
        "scale": 10,
        "region": aoi.geometry().bounds().getInfo(),
        "format": "GeoTIFF",
        "crs": "EPSG:4326"
    })

    output_file = f"Sentinel1_{date}.tiff"

    response = requests.get(url)
    if response.status_code == 200:
        with open(output_file, "wb") as f:
            f.write(response.content)
        print(f"Image downloaded as {output_file}")
    else:
        print(f"Failed to download image. HTTP Status: {response.status_code}")
except Exception as e:
    print(f"An error occurred: {e}")

Image downloaded as Sentinel1_2025-07-18.tiff
